In [1]:
#| default_exp restxl

In [2]:
from torch import inf

In [3]:
#| hide
import nbdev; nbdev.nbdev_export()

In [4]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [6]:
#| export
from rest.core import init_instance, generate
singleton, model_path = init_instance()

In [7]:
model_path = 'pelevin'

In [8]:
from transformers import GPT2Tokenizer, PreTrainedModel, PretrainedConfig
tokenizer = GPT2Tokenizer.from_pretrained("./tokenizer/rugpt3xl.tokenizer", local_files_only=True)


/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/xl/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_2048.json",
    seq_len=seq_length,
)
tokenizer = model.tokenizer
model.cuda()
model.eval();

[W socket.cpp:426] [c10d] The server socket cannot be initialized on [::]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [::ffff:127.0.0.1]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [::ffff:127.0.0.1]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [::ffff:127.0.0.1]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to

> initializing model parallel with size 1


In [10]:
model

RuGPT3XL(
  (model): FP16_Module(
    (module): GPT3Model(
      (word_embeddings): VocabParallelEmbedding()
      (position_embeddings): Embedding(2048, 2048)
      (embedding_dropout): Dropout(p=0.1, inplace=False)
      (transformer): GPT3ParallelTransformer(
        (layers): ModuleList(
          (0-23): 24 x GPT3ParallelTransformerLayer(
            (input_layernorm): FusedLayerNorm(torch.Size([2048]), eps=1e-05, elementwise_affine=True)
            (attention): GPT3ParallelSelfAttention(
              (query_key_value): ColumnParallelLinear()
              (attention_dropout): Dropout(p=0.1, inplace=False)
              (dense): RowParallelLinear()
              (output_dropout): Dropout(p=0.1, inplace=False)
            )
            (post_attention_layernorm): FusedLayerNorm(torch.Size([2048]), eps=1e-05, elementwise_affine=True)
            (mlp): GPT3ParallelMLP(
              (dense_h_to_4h): ColumnParallelLinear()
              (dense_4h_to_h): RowParallelLinear()
        

In [11]:
sum(p.numel() for p in model.parameters())

1315737600

In [12]:
#| export
import deepspeed, torch
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.float16,
                                 checkpoint=None,
                                 replace_with_kernel_inject=True)
model = ds_engine.module

[2023-05-27 16:46:33,745] [INFO] [logging.py:96:log_dist] [Rank -1] DeepSpeed info: version=0.9.2, git-hash=unknown, git-branch=unknown
[2023-05-27 16:46:33,747] [WARNING] [config_utils.py:69:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead
[2023-05-27 16:46:33,748] [INFO] [logging.py:96:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [13]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [14]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 5.3 s, sys: 165 ms, total: 5.46 s
Wall time: 5.42 s


[' полный шакал", - вынес я приговор вслух, не глядя в их сторону. Зря, однако, сказал - они по-прежнему пожирали друг друга глазами. "Жрите!" - еще раз мысленно предложил я им и пошел прочь.',
 '… ты как был кот в мешке, так и остался. Это – серый кардинал. Качаешь миллионы из воздуха. Вроде бы и контроля никакого, но внутри этой конторы одно жулье сидит.',
 ' хуйня полная» Это уже в камере. Впихнув в мусорный бак последнюю урну, в камеру всунули молодого человека, затолкали внутрь все камеры, заперли, и барак пошел в тюрьму.',
 ' — нет… А там еще фазан есть… Муй зашибенный. С колокольчиками. Называется «Петухи поют»… Или все-таки не «петухи?» Ведь петух.']

In [15]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])

/usr/local/lib/python3.8/dist-packages/transformers/generation/logits_process.py:660: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /opt/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:589.)
  torch.sparse.LongTensor(banned_mask.t(), indices, scores.size())



- Меня Федором зовут.
– Кто это?– переспросила Мария.– Федор? Какой Федор, тут никогда не было никакого Федора! И не будет! Уходи отсюда!

CPU times: user 1.13 s, sys: 0 ns, total: 1.13 s
Wall time: 1.09 s
